In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (UniDL4BioPep)

This notebook processes and standardizes the **UniDL4BioPep** toxicity dataset into a unified and consistent format suitable for downstream analysis and machine learning tasks. The source provides peptide sequences and labels across multiple files and formats (CSV and Excel), including training and test splits from different dataset releases.

- **Toxic effect / endpoint:** toxic
- **Source:** UniDL4BioPep
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads peptide sequences and labels** from multiple CSV and Excel files corresponding to different dataset versions.
- **Concatenates all sources** into a single unified dataset with a canonical schema:
  - `sequence`: peptide amino-acid sequence
  - `label`: toxicity label
- **Performs duplicate sequence quality control**:
  - consolidates identical sequences with consistent labels,
  - flags sequences with conflicting annotations as erroneous.
- **Generates dataset-level metadata** using the centralized raw-data description spreadsheet.
- **Exports the curated dataset and metadata** to the standardized output directory.

In [2]:
name_source = "UniDL4BioPep"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_test_580_17 = pd.read_excel(f"{PATH_INPUT}/{name_source}/17. Toxicity 2021 Dataset/test_580.xlsx")

In [4]:
df_train_3284_17 = pd.read_excel(f"{PATH_INPUT}/{name_source}/17. Toxicity 2021 Dataset/train_3284.xlsx")

In [5]:
df_test_580 = pd.read_csv(f"{PATH_INPUT}/{name_source}/test_580.csv")

In [6]:
df_train_3284 = pd.read_csv(f"{PATH_INPUT}/{name_source}/train_3284.csv")

- Concatenate dataset

In [7]:
df_unidl4biopep = (
    pd.concat([df_train_3284_17, df_test_580_17,
               df_test_580, df_train_3284], ignore_index=True)
    [["sequence", "label"]]
)
df_unidl4biopep.shape

(7728, 2)

- Checking duplicates

In [8]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_unidl4biopep, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [9]:
df_full.shape

(3864, 2)

In [10]:
df_errors.shape

(0, 1)

- Working with metada

In [11]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [12]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_unidl4biopep)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Dynamic',
 'license': 'MIT',
 'year of publication': 2023,
 'last update date': datetime.datetime(2024, 11, 22, 0, 0),
 'download date': '2025-05-01 00:00:00;2025-04-01 00:00:00',
 'file format': 'xlsx',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Previously published model dataset;Reported in literature',
 'repository or server': 'https://github.com/dzjxzyd/UniDL4BioPep',
 'publication': 'https://academic.oup.com/bib/article/24/3/bbad135/7107929',
 'number_of_raw_sequences': 7728,
 'number_of_sequences_retained': 3864,
 'number_of_positive_sequences': 1932,
 'number_of_negative_sequences': 1932,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [13]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [14]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)